# 01 — Renderer validation (ref-port, deterministic)

Ports `yanhong-lbh/text_or_pixels:text_to_image.py`: LaTeX-escape → Tectonic PDF → `pdf2image` @300 DPI → fixed `W×H` + font-size search to `fill_ratio≈0.85`, single-page assert. Falls back to PIL direct-draw when tectonic/poppler are absent (common on locked Kaggle images) — fallback is labeled as such in logs.

In [ ]:
import io, os, re, shutil, subprocess, tempfile, textwrap, unicodedata
from PIL import Image, ImageDraw, ImageFont

HAVE_TECTONIC = shutil.which("tectonic") is not None
try:
    from pdf2image import convert_from_path
    HAVE_PDF2IMAGE = True
except ImportError:
    HAVE_PDF2IMAGE = False
print(f"tectonic={HAVE_TECTONIC} pdf2image={HAVE_PDF2IMAGE}")

LATEX_ESCAPES = {"&": r"\&", "%": r"\%", "$": r"\$", "#": r"\#", "_": r"\_", "{": r"\{", "}": r"\}", "~": r"\textasciitilde{}", "^": r"\textasciicircum{}", "<": r"\textless{}", ">": r"\textgreater{}"} 
TYPO_FIX = {"\u2019": "'", "\u2018": "'", "\u201c": '"', "\u201d": '"', "\u2013": "-", "\u2014": "--", "\u2026": "..."}

def normalize_text(t: str) -> str:
    t = unicodedata.normalize("NFKC", t or "")
    t = t.replace("\r\n", "\n").replace("\r", "\n")
    for a, b in TYPO_FIX.items():
        t = t.replace(a, b)
    t = re.sub(r"[ \t]+", " ", t)
    return t.strip()

def escape_latex(t: str) -> str:
    t = t.replace("\\", r"\textbackslash{}")
    for ch, rep in LATEX_ESCAPES.items():
        t = t.replace(ch, rep)
    return t

def bbox_fill_ratio(img: Image.Image, bg_thresh: int = 240) -> float:
    gray = img.convert("L")
    mask = gray.point(lambda p: 0 if p > bg_thresh else 255, mode="1")
    bbox = mask.getbbox()
    if not bbox:
        return 0.0
    return ((bbox[2] - bbox[0]) * (bbox[3] - bbox[1])) / (gray.width * gray.height)

def render_pil(text: str, width=800, font_size=18, line_spacing=6, wrap=80, pad=30) -> Image.Image:
    """Deterministic PIL fallback. Logs renderer='pil-fallback' so results are never mixed silently."""
    text = normalize_text(text)
    try:
        font = ImageFont.truetype("arial.ttf", font_size)
    except IOError:
        font = ImageFont.load_default()
    lines: list[str] = []
    for para in text.split("\n\n"):
        lines.extend(textwrap.wrap(para, width=wrap))
        lines.append("")
    lh, H = font_size + line_spacing, 0
    H = 2 * pad + max(len(lines) * lh, 100)
    img = Image.new("RGB", (width, H), "white")
    d = ImageDraw.Draw(img)
    y = pad
    for ln in lines:
        if ln:
            d.text((pad, y), ln, fill="black", font=font)
        y += lh
    return img

In [ ]:
# Cell 2 — validation: render short/med/long synthetic contexts at 2 canvas sizes, assert + save proofs
import json

DISTRACTOR = ("The thermal radiator on the secondary boom radiates excess heat into cold space. "
              "Star tracking sensor arrays refresh coordinate quaternions at thirty hertz. ")

def make_context(target_words: int, seed: int = 7):
    import random
    rng = random.Random(seed)
    code = f"CIPHER-{rng.randint(1000, 9999)}-OMEGA"
    needle = f"Special authorization key {code} was recorded in emergency protocol ledger section 9."
    paras, n = [], 0
    while n < target_words:
        p = " ".join(rng.choice([DISTRACTOR]) for _ in range(6))
        paras.append(p)
        n += len(p.split())
    mid = len(paras) // 2
    w = paras[mid].split(); w.insert(len(w) // 2, needle)
    paras[mid] = " ".join(w)
    return "\n\n".join(paras), code

CANVASES = [(600, 1000), (1100, 1400)]  # 2 visual budgets for the validation gate
os.makedirs("/kaggle/working/results/raw", exist_ok=True)
report = []
for words, tag in [(350, "short~500tok"), (1500, "med~2ktok"), (3000, "long~4ktok")]:
    ctx, code = make_context(words)
    for W, H in CANVASES:
        img = render_pil(ctx, width=W)  # swap to tectonic path when available
        ratio = bbox_fill_ratio(img)
        # no-crop heuristic: content bbox must not touch image edge (margin >= 4px)
        gray = img.convert("L")
        bbox = gray.point(lambda p: 0 if p > 240 else 255, mode="1").getbbox() or (0, 0, 0, 0)
        margin = min(bbox[0], bbox[1], img.width - bbox[2], img.height - bbox[3])
        ok = ratio > 0.05 and margin >= 4
        fn = f"/kaggle/working/results/raw/val_{tag}_{W}x{img.height}.png"
        img.save(fn)
        report.append({"context": tag, "canvas": f"{W}x{H}", "actual": f"{img.width}x{img.height}",
                         "fill_ratio": round(ratio, 3), "edge_margin_px": margin, "pass": bool(ok), "file": fn})
        print(report[-1])
assert all(r["pass"] for r in report), "VALIDATION FAILED — fix renderer before 02"
print(f"\nGATE PASS: {len(report)} renders uncropped | renderer=pil-fallback (label in 02 logs)")